# Подготовка внутри разбиения и доступность признаков

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Прослеживать обучение преобразований в каждом разбиении и проверять доступность признаков.

## Каждый блок — отдельный маленький эксперимент

Внутри перекрёстной проверки преобразователь должен учиться только по внутреннему train. Даже если тестовые ответы не использованы, среднее или медиана по проверочным признакам передают информацию об их распределении. Порядок такой: сначала разделите данные, затем вызовите `fit` для преобразователя и модели на обучающей части. Проверочную часть преобразуйте через `transform`, сохраняя уже найденные параметры.

В [уроке о конвейерах](../../../python/ml-tools/pipelines/) мы собирали шаги подготовки. Теперь проследим, по каким строкам каждый шаг обучается в разных блоках проверки: готовый конвейер ещё нужно включить в правильный протокол.

Pipeline объединяет эти шаги. ColumnTransformer позволяет отдельно готовить разные столбцы. Но конвейер не знает, что «дата закрытия заявки» появляется после прогнозируемого решения: доступность каждого признака нужно проверить самостоятельно.

## Разобранный пример: разные средние

В первом разбиении обучающие значения [0,2], их среднее 1. Во втором — [8,10], их среднее 9. Проверочный объект 4 при первом центрировании станет 3, при втором −5. Если использовать общее среднее 5 в обоих случаях, в подготовку попадут данные за пределами соответствующей обучающей части.

После выбора настроек преобразователь заново обучают на всех данных, которые разрешены для финального обучения. Его параметры могут отличаться от найденных в отдельных разбиениях — это нормально. К итоговому тесту применяют сохранённое правило без нового вызова `fit`.

## Проверяем доступность

Составьте для столбцов таблицу «источник — момент появления — можно ли использовать». Идентификатор группы нужен для разбиения, ответ — для обучения, время — для разделения прошлого и будущего. Эти столбцы не обязательно нужно передавать модели как признаки. Перед запуском проверьте форму и порядок колонок, а также то, что каждый объект проверки получает ровно один прогноз.

Завершим конвейер уже знакомым методом ближайшего соседа. Здесь нас интересует не качество классификации, а среднее, которое масштабирование выучило в каждом блоке. Метки 0 и 1 нужны, чтобы обучить полный конвейер.

В ноутбуке clone создаёт новый необученный объект с теми же настройками. Это предотвращает перенос выученного состояния между блоками. Мы отдельно смотрим mean_ — обученное среднее StandardScaler, чтобы проверить, не обучался ли преобразователь сразу на всех данных.

У конвейера `named_steps` — словарь шагов по заданным именам. Поэтому `model.named_steps["scale"].mean_` показывает среднее, выученное конкретным масштабированием в этом запуске.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
base = Pipeline([("scale", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=1))])
learned_means = []
for values in ([[0.], [2.]], [[8.], [10.]]):
    model = clone(base)
    model.fit(values, [0., 1.])
    learned_means.append(model.named_steps["scale"].mean_[0])
assert np.allclose(learned_means, [1., 9.])
print(learned_means)


## Типичные ошибки

Не вызывайте fit_transform на validation. Не считайте Pipeline доказательством отсутствия смысловой утечки.

## Задачи

### 1. Базовая

По каким данным в каждом разбиении находят значения для заполнения пропусков?

<details>
<summary>Подсказка</summary>

После выбора внутреннего train.

</details>



### 2. Базовая

Делает ли transform новое обучение?

<details>
<summary>Подсказка</summary>

Он использует сохранённое состояние.

</details>



### 3. Прикладная

Среднее train равно 7, объект validation равен 10. Найдите центрированное значение.

<details>
<summary>Подсказка</summary>

Используйте обучающее среднее.

</details>



### 4. Прикладная

Средние обучающих частей двух разбиений равны 1 и 9. Должны ли они совпадать?

<details>
<summary>Подсказка</summary>

Обучающие строки разные.

</details>



### 5. Олимпиадная

Зачем создавать новую модель для каждого разбиения?

<details>
<summary>Подсказка</summary>

Иначе может сохраниться старое состояние.

</details>



### 6. Олимпиадная

Защитит ли конвейер от столбца из будущего?

<details>
<summary>Подсказка</summary>

Он не знает временного смысла источника.

</details>



## Итоги и самопроверка

Проследите, как один объект проходит подготовку и модель в двух разбиениях. Как обнаружить, что параметры масштабирования вычислили сразу по всем данным?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Только по обучающей части этого разбиения.

### Задача 2

Нет. Параметры подготовки остаются прежними.

### Задача 3

10−7=3; среднее validation не нужно.

### Задача 4

Нет. Совпадение не является целью, важно, какие строки использованы.

### Задача 5

Он исключает случайное продолжение обучения или использование параметров предыдущего разбиения.

### Задача 6

Нет. Нужно удалить или пересчитать признак по доступному временному срезу.